# Lab 2: การทำความสะอาดข้อมูลดิบจากเซนเซอร์ IoT

ไฟล์ข้อมูล: `iot_sensor_raw_data_extended.csv`

In [2]:
import pandas as pd
import numpy as np

# 1. โหลดข้อมูล
df_raw = pd.read_csv('/content/iot_sensor_raw_data_extended.csv')

# ตรวจสอบข้อมูลเบื้องต้น
print("--- ข้อมูลเบื้องต้น ---")
print(df_raw.info())

# ตรวจสอบค่าว่าง (Missing Values)
print("\n--- จำนวน Missing Values ในแต่ละคอลัมน์ ---")
print(df_raw.isnull().sum())

# ตรวจสอบข้อมูลซ้ำ (Duplicates)
# (device_id และ timestamp เดียวกัน)
duplicate_count = df_raw.duplicated(subset=['device_id', 'timestamp'], keep=False).sum()
print(f"\nจำนวนข้อมูลที่มีคู่ซ้ำ (Same Device & Timestamp): {duplicate_count} แถว")

display(df_raw.head())

--- ข้อมูลเบื้องต้น ---
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 480 entries, 0 to 479
Data columns (total 5 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   timestamp  480 non-null    object 
 1   device_id  478 non-null    object 
 2   temp_c     473 non-null    float64
 3   motion     476 non-null    object 
 4   battery    476 non-null    float64
dtypes: float64(2), object(3)
memory usage: 18.9+ KB
None

--- จำนวน Missing Values ในแต่ละคอลัมน์ ---
timestamp    0
device_id    2
temp_c       7
motion       4
battery      4
dtype: int64

จำนวนข้อมูลที่มีคู่ซ้ำ (Same Device & Timestamp): 50 แถว


,timestamp,device_id,temp_c,motion,battery
0,2026-07-12 14:50:00,SN-9982,28.3,false,88.0
1,2026-07-12 14:50:00,SN-9983,28.2,true,94.0
2,2026-07-12 14:50:00,SN-9984,29.7,true,76.0
3,2026-07-12 14:50:00,SN-9985,27.8,true,81.0
4,2026-07-12 14:50:00,SN-9986,30.7,true,69.0


In [4]:
# คัดลอก DataFrame มาเพื่อทำความสะอาด
df_cleaned = df_raw.copy()

# เก็บจำนวนแถวก่อนทำความสะอาด
rows_before = len(df_cleaned)

# 1. ลบข้อมูลซ้ำ (Duplicate)
df_cleaned = df_cleaned.drop_duplicates(subset=['device_id', 'timestamp'], keep='first')
duplicates_removed = rows_before - len(df_cleaned)

# 2. จัดการ Outliers และ Missing Values (เปลี่ยนค่าที่ผิดกฎเป็น NaN)
# กฎ: 0 <= temp_c <= 50
df_cleaned.loc[(df_cleaned['temp_c'] < 0) | (df_cleaned['temp_c'] > 50), 'temp_c'] = np.nan

# กฎ: 0 <= battery <= 100
df_cleaned.loc[(df_cleaned['battery'] < 0) | (df_cleaned['battery'] > 100), 'battery'] = np.nan

# 3. เพิ่มคอลัมน์ data_quality_status
def check_status(row):
    if pd.isna(row['device_id']):
        return 'Error: Missing Device ID'
    if pd.isna(row['temp_c']) or pd.isna(row['battery']) or pd.isna(row['motion']):
        return 'Warning: Missing or Outlier Data'
    return 'Clean'

df_cleaned['data_quality_status'] = df_cleaned.apply(check_status, axis=1)

# 4. บันทึกไฟล์ที่สะอาดแล้ว
df_cleaned.to_csv('cleaned_iot_data.csv', index=False)

# 5. สรุปผล
print(f"--- สรุปการทำความสะอาด ---")
print(f"จำนวนแถวก่อนทำ: {rows_before}")
print(f"จำนวนแถวหลังทำ (หลังลบ duplicate): {len(df_cleaned)}")
print(f"จำนวนข้อมูลซ้ำที่ถูกลบ: {duplicates_removed}")
print(f"\nสถานะคุณภาพข้อมูล:")
print(df_cleaned['data_quality_status'].value_counts())

display(df_cleaned.head(10))

--- สรุปการทำความสะอาด ---
จำนวนแถวก่อนทำ: 480
จำนวนแถวหลังทำ (หลังลบ duplicate): 455
จำนวนข้อมูลซ้ำที่ถูกลบ: 25

สถานะคุณภาพข้อมูล:
data_quality_status
Clean                               426
Error: Missing Device ID              2
Name: count, dtype: int64


,timestamp,device_id,temp_c,motion,battery,data_quality_status
0,2026-07-12 14:50:00,SN-9982,28.3,false,88.0,Clean
1,2026-07-12 14:50:00,SN-9983,28.2,true,94.0,Clean
2,2026-07-12 14:50:00,SN-9984,29.7,true,76.0,Clean
3,2026-07-12 14:50:00,SN-9985,27.8,true,81.0,Clean
4,2026-07-12 14:50:00,SN-9986,30.7,true,69.0,Clean
5,2026-07-12 14:50:10,SN-9982,28.4,true,88.0,Clean
6,2026-07-12 14:50:10,SN-9983,28.1,true,94.0,Clean
7,2026-07-12 14:50:10,SN-9984,29.6,true,76.0,Clean
8,2026-07-12 14:50:10,SN-9985,27.7,false,81.0,Clean
9,2026-07-12 14:50:10,SN-9986,31.0,false,69.0,Clean


### สรุป: ทำไมจึงไม่ควรลบ Raw Data ต้นฉบับ

การเก็บรักษาข้อมูลดิบ (Raw Data) มีความสำคัญอย่างยิ่งในกระบวนการทำ Data Engineering และ Data Science ด้วยเหตุผลดังนี้:

1. **Data Lineage & Traceability**: เพื่อให้สามารถตรวจสอบย้อนกลับ (Audit Trail) ได้ว่าข้อมูลมีการเปลี่ยนแปลงหรือถูกประมวลผลอย่างไรตั้งแต่ต้นทางจนถึงปลายทาง
2. **Error Recovery & Re-processing**: ในกรณีที่พบว่าอัลกอริทึมหรือกฎการทำความสะอาดข้อมูล (Data Quality Rules) ในปัจจุบันมีข้อผิดพลาด เราสามารถนำข้อมูลดิบกลับมาประมวลผลใหม่ด้วยกฎที่ถูกต้องได้
3. **Compliance & Legal Requirements**: ในหลายองค์กรมีกฎระเบียบหรือข้อบังคับทางกฎหมายที่ระบุให้ต้องจัดเก็บข้อมูลต้นฉบับไว้เพื่อการตรวจสอบในอนาคต
4. **Flexibility**: ข้อมูลที่ถูกมองว่าเป็น Error หรือ Outlier ในวันนี้ อาจจะเป็นข้อมูลที่มีค่าสำหรับการวิเคราะห์ในมุมมองใหม่ๆ ในวันหน้า การลบทิ้งจะทำให้เราสูญเสียโอกาสนั้นไปถาวร